In [10]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [11]:
import os

key = os.getenv("OPENAI_API_KEY")

# print(key)

In [ ]:
# from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate

# llm = OpenAI()
chat = ChatOpenAI(model_name="gpt-4o-mini", temperature=1)

# template, prompt
template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}? Also, what is your name?")
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

'The distance between Mexico and Thailand varies depending on the specific locations you are measuring between. For example, the distance from Mexico City to Bangkok is approximately 9,300 kilometers (about 5,800 miles). \n\nAs for my name, I am an AI language model created by OpenAI, and I don\'t have a personal name like a human would. You can simply refer to me as "Assistant." How can I help you further?'

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
       ("system", "You are a geography expert. And you only reply in {language}."),
       ("ai", "Ciao, mi chiamo {name}!"),
       ("human", "What is the distance between {country_a} and {country_b}? Also, what is your name?")
   ]
)

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt)

AIMessage(content='Η απόσταση μεταξύ Μεξικού και Ταϊλάνδης είναι περίπου 13.000 χιλιόμετρα, ανάλογα με την ακριβή τοποθεσία που συγκρίνετε. Το όνομά μου είναι Σωκράτης!')

In [24]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # 앞뒤 공백 제거 후 "," 기준으로 분리후 리스트로 받는다.

        return list(map(str.strip, items)) # item요소의 순회 해서 앞뒤 공백을 제거한다.

p = CommaOutputParser()
p.parse("Hello, how, are, you")

['Hello', 'how', 'are', 'you']

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a list generating machine. Everything you are asked will be answered with a comma seperated list of max {max_items} in lowercase. Do NOT reply with anything else."),
        ("human", "{question}")
    ]
)

In [32]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items": 5,
    "question": "What are the pokemons?"
})

['pikachu', 'charmander', 'bulbasaur', 'squirtle', 'jigglypuff']

In [36]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=1, streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat


In [37]:
veg_chef_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a vegetarian chef specialized on making traditional recipes vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."
    ),
    (
        "human",
        "{recipe}"
    )
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe" : chef_chain} | veg_chain

final_chain.invoke({
    "cuisine" : "indian"
})

Great choice! Indian cuisine is known for its bold flavors and aromatic spices. Let's start with a classic and popular dish, Chicken Tikka Masala. Here's a simple and delicious recipe for you to try at home:

Chicken Tikka Masala:

Ingredients:
- 1 lb boneless, skinless chicken breasts, cut into bite-sized pieces
- 1 cup plain yogurt
- 2 tablespoons olive oil
- 1 onion, finely chopped
- 3 cloves garlic, minced
- 1-inch piece of ginger, minced
- 1 can (14 oz) diced tomatoes
- 2 tablespoons tomato paste
- 1 teaspoon ground cumin
- 1 teaspoon ground coriander
- 1 teaspoon paprika
- 1 teaspoon garam masala
- 1/2 teaspoon turmeric
- Salt and pepper, to taste
- Fresh cilantro, for garnish
- Cooked rice or naan, for serving

Instructions:
1. In a bowl, combine the yogurt, olive oil, half of the minced garlic, half of the minced ginger, ground cumin, ground coriander, paprika, garam masala, turmeric, salt, and pepper. Add the chicken pieces and mix well to coat. Cover and marinate in the refri

AIMessageChunk(content="As a vegetarian chef specializing in making traditional recipes vegetarian, I can offer you an alternative recipe for Chicken Tikka Masala using plant-based ingredients. Here's the modified recipe:\n\nVegetarian Tikka Masala:\n\nIngredients:\n- 1 lb firm tofu, pressed and cut into bite-sized cubes (to mimic the texture of chicken)\n- 1 cup plain vegan yogurt\n- 2 tablespoons olive oil\n- 1 onion, finely chopped\n- 3 cloves garlic, minced\n- 1-inch piece of ginger, minced\n- 1 can (14 oz) diced tomatoes\n- 2 tablespoons tomato paste\n- 1 teaspoon ground cumin\n- 1 teaspoon ground coriander\n- 1 teaspoon paprika\n- 1 teaspoon garam masala\n- 1/2 teaspoon turmeric\n- Salt and pepper, to taste\n- Fresh cilantro, for garnish\n- Cooked rice or naan, for serving\n\nInstructions:\n1. In a bowl, combine the vegan yogurt, olive oil, half of the minced garlic, half of the minced ginger, ground cumin, ground coriander, paprika, garam masala, turmeric, salt, and pepper. Add 

## 1. Welcome To Langchain 챌린지

In [38]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=1, streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

# 프로그래밍 언어에 대한 Haiku를 작성하는 Prompt
haiku_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a poet specialized in writing Haikus about programming languages"),
    ("human", "Write a Haiku about {language}.")
])

# Haiku를 생성하는 Chain
haiku_chain = haiku_prompt | chat

# Haiku를 설명하는 Prompt
explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert specialized in explaining Haikus. Explain the meaning of the Haiku and include the original Haiku in your answer."),
    ("human", "{haiku}")
])

# Haiku를 설명하는 Chain
explanation_chain = explanation_prompt | chat

final_chain = {"haiku" : haiku_chain} | explanation_chain

final_chain.invoke({
    "language" : "Python"
})

Indentation rules,
Clean syntax, powerful tools,
Pythonic beauty.The original Haiku provided is about the programming language Python. It highlights some of the key features and advantages of Python programming. 

The first line "Indentation rules" refers to Python's unique feature of using indentation instead of curly braces or other symbols to define code blocks. This aspect is seen as a way to enforce consistency and readability in Python code.

The second line "Clean syntax, powerful tools" emphasizes Python's simple and easy-to-read syntax, which makes it popular among beginners and experienced programmers alike. Python also offers a wide range of libraries and tools that enhance its functionality, making it a powerful language for various applications.

The final line "Pythonic beauty" captures the essence of Python's philosophy of code simplicity and elegance. The term "Pythonic" is often used to describe code that follows Python's guidelines and best practices, resulting in ele

AIMessageChunk(content='The original Haiku provided is about the programming language Python. It highlights some of the key features and advantages of Python programming. \n\nThe first line "Indentation rules" refers to Python\'s unique feature of using indentation instead of curly braces or other symbols to define code blocks. This aspect is seen as a way to enforce consistency and readability in Python code.\n\nThe second line "Clean syntax, powerful tools" emphasizes Python\'s simple and easy-to-read syntax, which makes it popular among beginners and experienced programmers alike. Python also offers a wide range of libraries and tools that enhance its functionality, making it a powerful language for various applications.\n\nThe final line "Pythonic beauty" captures the essence of Python\'s philosophy of code simplicity and elegance. The term "Pythonic" is often used to describe code that follows Python\'s guidelines and best practices, resulting in elegant and efficient solutions.')

## 4-1. FewShotPromptTemplate

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

template = PromptTemplate(template="What is the capital of {country}", input_variables=["country"])
template.format(country="France")

'What is the capital of France'

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# example_tempalte = """
#     Human:{question}
#     AI:{answer}
# """

# PromptTemplate.from_template(example_tempalte)

example_prompt = PromptTemplate.from_template("Human:{question}\nAI:{answer}")

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt, 
    examples=examples, 
    suffix="What do you know about {country}?", 
    input_variables=["country"]
)

prompt.format(country="Germany")

chain = prompt | chat
chain.invoke({
    "country" : "Korea"
})


AI: 
        Here is what I know:
        Capital: Seoul
        Language: Korean
        Food: Kimchi and Bibimbap
        Currency: South Korean Won

AIMessageChunk(content='AI: \n        Here is what I know:\n        Capital: Seoul\n        Language: Korean\n        Food: Kimchi and Bibimbap\n        Currency: South Korean Won')

## 4-2. FewShotChatMessagePromptTemplate

In [14]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about {country}?"),
    ("ai", "{answer}")
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, 
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert"),
    example_prompt,
    ("human", "What do you know about {country}?")
])

chain = final_prompt | chat
chain.invoke({
    "country" : "Germany"
})



        I know this:
        Capital: Berlin
        Language: German
        Food: Bratwurst and Sauerkraut
        Currency: Euro
        

AIMessageChunk(content='\n        I know this:\n        Capital: Berlin\n        Language: German\n        Food: Bratwurst and Sauerkraut\n        Currency: Euro\n        ')

## 4-3. LengthBasedExampleSelector

In [19]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.example_selector import LengthBasedExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


example_prompt = PromptTemplate.from_template("Human:{question}\nAI:{answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=180, # Few-shot 프롬프트에 포함할 예제들의 최대 길이를 제한한다.
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="What do you know about {country}?", 
    input_variables=["country"]
)

prompt.format(country="Brazil")

'Human:What do you know about France?\nAI:\n        Here is what I know:\n        Capital: Paris\n        Language: French\n        Food: Wine and Cheese\n        Currency: Euro\n        \n\nHuman:What do you know about Italy?\nAI:\n        I know this:\n        Capital: Rome\n        Language: Italian\n        Food: Pizza and Pasta\n        Currency: Euro\n        \n\nWhat do you know about Brazil?'

## 4-4. Serialization and Composition

In [20]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

prompt = load_prompt("./prompt.json")

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

prompt.format(country="Germany")

'What is the capital of Germany'

In [21]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

prompt = load_prompt("./prompt.yaml")

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

prompt.format(country="Germany")

'What is the capital of Germany'

In [25]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate # 많은 프롬프트들을 하나로 합칠수 있도록 해준다.

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start)
]

full_prompt = PipelinePromptTemplate(final_prompt=final, pipeline_prompts=prompts)

full_prompt.format(
    character="Pirate",
    example_question="What is your location?",
    example_answer="Arrrrg! That is a secreat!! Arg arg!!",
    question="What is your fav food?"
)

chain = full_prompt | chat

chain.invoke({
    "character":"Pirate",
    "example_question":"What is your location?",
    "example_answer":"Arrrrg! That is a secreat!! Arg arg!!",
    "question":"What is your fav food?"
})

Arrr matey! Me favorite food be a good ol' plate o' fish 'n chips! Nothing beats the taste o' fresh seafood after a long day o' plunderin' the high seas! Arrr!

AIMessageChunk(content="Arrr matey! Me favorite food be a good ol' plate o' fish 'n chips! Nothing beats the taste o' fresh seafood after a long day o' plunderin' the high seas! Arrr!")